In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import datasets, transforms, models
import numpy as np
from sklearn.metrics import classification_report
from sklearn.metrics import classification_report
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')

# Этап 1. Загрузка данных

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_dir = 'ogyeiv2'
full_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'train'))

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transform
val_dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(full_dataset.classes)

# Этап 2. Создание модели

weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)
print("Модель готова")

# Этап 3. Обучение

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

num_epochs = 1
train_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    train_losses.append(running_loss / len(train_loader))
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_acc = 100 * correct / total
    val_accuracies.append(val_acc)
    
    print(f'Эпоха {epoch+1}: Loss={train_losses[-1]:.4f}, Val Acc={val_acc:.2f}%')

torch.save(model.state_dict(), 'meds_classifier.pt')
print("Модель сохранена")

# Этап 4. Оценка качества

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

accuracy = np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
print(f'\nОбщая точность(accuracy): {accuracy:.2%}')

if accuracy > 0.75:
    print("Целевая точночть выше 75% достигнута!")
else:
    print(f"Текущая точность {accuracy:.2%} ниже целевых 75%")

print('Метрики для всех классов')

class_names = full_dataset.classes
report_dict = classification_report(
    all_labels, 
    all_preds, 
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

df = pd.DataFrame(report_dict).transpose()
df = df[~df.index.isin(['accuracy', 'macro avg', 'weighted avg'])]

print(df[['precision', 'recall', 'f1-score', 'support']].to_string())

df_sorted = df.sort_values('f1-score', ascending=False)

print('\n5 классов с лучшим F1-SCORE:')
for i, (idx, row) in enumerate(df_sorted.head(5).iterrows(), 1):
    print(f"   {i}. {idx[:40]:<40} F1={row['f1-score']:.3f} (примеров: {row['support']:.0f})")

print('\n5 классов с худшим F1-SCORE:')
for i, (idx, row) in enumerate(df_sorted.tail(5).iterrows(), 1):
    print(f"   {i}. {idx[:40]:<40} F1={row['f1-score']:.3f} (примеров: {row['support']:.0f})")


worst_classes = df_sorted.tail(5)
print('\n1. На каких 5 классах модель ошибается чаще всего?')
for i, (idx, row) in enumerate(worst_classes.iterrows(), 1):
    error_rate = 1 - row['f1-score']  # приблизительно
    print(f"   {i}. {idx[:40]:<40} - ошибок: ~{error_rate:.2%}")

print('\n2. Почему модель может ошибаться на этих классах?')
print('   • Мало обучающих примеров:')
small_support = worst_classes[worst_classes['support'] < 20]
for idx, row in small_support.iterrows():
    print(f'     - {idx[:30]} (всего {row["support"]:.0f} примеров)')
print('   • Визуальное сходство с другими классами')
print('   • Разные ракурсы и освещение на фото')

best_classes = df_sorted.head(5)
print('\n3. На каких классах модель ошибается меньше всего?')
for i, (idx, row) in enumerate(best_classes.iterrows(), 1):
    error_rate = 1 - row['f1-score']
    print(f"   {i}. {idx[:40]:<40} - ошибок: {error_rate:.2%}")

print('\n4. Почему эти классы распознаются хорошо?')
print('   • Достаточно обучающих примеров:')
for idx, row in best_classes.head(3).iterrows():
    print(f'     - {idx[:30]} ({row["support"]:.0f} примеров)')
print('   • Уникальные визуальные признаки')
print('   • Четкие фото с минимальными вариациями')

print('\n5. Как можно улучшить точность классификатора?')
print('   • Добавить больше аугментации данных')
print('   • Увеличить число эпох обучения')
print('   • Разморозить больше слоев для тонкой настройки')
print('   • Добавить примеров для проблемных классов:')
for idx, row in worst_classes.head(3).iterrows():
    print(f'     - {idx[:30]} (сейчас {row["support"]:.0f} примеров)')

print('\n6. Как ещё можно проанализировать результаты?')
print('   • Построить матрицу ошибок')
print('   • Посмотреть на неправильно классифицированные изображения')

Устройство: cpu
Модель готова
Эпоха 1: Loss=4.1928, Val Acc=24.42%
Модель сохранена

Общая точность(accuracy): 23.78%
Текущая точность 23.78% ниже целевых 75%
Метрики для всех классов
                                  precision    recall  f1-score  support
acc_long_600_mg                    1.000000  0.666667  0.800000      6.0
advil_ultra_forte                  0.500000  1.000000  0.666667      9.0
akineton_2_mg                      0.444444  0.571429  0.500000      7.0
algoflex_forte_dolo_400_mg         1.000000  0.166667  0.285714      6.0
algoflex_rapid_400_mg              0.000000  0.000000  0.000000      6.0
algopyrin_500_mg                   0.000000  0.000000  0.000000      8.0
ambroxol_egis_30_mg                0.333333  0.285714  0.307692      7.0
apranax_550_mg                     0.000000  0.000000  0.000000      4.0
aspirin_ultra_500_mg               0.263158  1.000000  0.416667      5.0
atoris_20_mg                       0.000000  0.000000  0.000000      6.0
atorvastatin_